# Experimento principal

## Imports necesarios

In [9]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from joblib import Parallel, delayed
import scipy.stats as stats
import scikit_posthocs as sp
import seaborn as sns
import matplotlib.pyplot as plt


## Funciones 

In [10]:
def procesar_dataset(archivo, n_splits=10, n_repeats=1):
    """
    Procesa el dataset usando SVM lineal.
    Soporta Validación Cruzada Clásica (1 repetición) o Repetida (N repeticiones).
    """
    if not os.path.exists(archivo):
        return archivo, None, None, f"[ERROR] Archivo no encontrado: {archivo}"

    df = pd.read_csv(archivo)
    
    # 1. Eliminar variables de sesgo global (Data Leakage)
    columnas_sesgo = ['P_0', 'P_1', 'P_2', 'mu_2', 'sigma_2']
    cols_a_borrar = ['clase'] + [c for c in columnas_sesgo if c in df.columns]
    
    X = df.drop(columns=cols_a_borrar).values
    y = df['clase'].values

    metricas = {
        'acc_train': [], 
        'acc_test': [], 
        'recall_test': [], 
        'f1_test': [], 
        'auc_test': []
    }
    
    # 2. Configurar la Validación Cruzada Estratificada Repetida
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

    fold = 1
    for train_idx, test_idx in rskf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        svm = SVC(kernel='linear', random_state=42)
        svm.fit(X_train, y_train)

        y_pred_train = svm.predict(X_train)
        y_pred_test = svm.predict(X_test)
        y_scores_test = svm.decision_function(X_test)

        metricas['acc_train'].append(accuracy_score(y_train, y_pred_train))
        metricas['acc_test'].append(accuracy_score(y_test, y_pred_test))
        metricas['recall_test'].append(recall_score(y_test, y_pred_test))
        metricas['f1_test'].append(f1_score(y_test, y_pred_test))
        metricas['auc_test'].append(roc_auc_score(y_test, y_scores_test))

        fold += 1

    # 3. Empaquetar resultados
    total_iteraciones = n_splits * n_repeats
    df_metricas = pd.DataFrame(metricas)
    df_metricas.index = [f"Iteracion_{i+1}" for i in range(total_iteraciones)]
    
    # Añadir el nombre del esquema para facilitar la combinación posterior
    nombre_esquema = archivo.replace('.csv', '').replace('radiomics_', '')
    df_metricas.insert(0, 'Esquema', nombre_esquema)

    stats = pd.DataFrame({
        'Media': df_metricas.iloc[:, 1:].mean(),
        'Mediana': df_metricas.iloc[:, 1:].median(),
        'Desv. Est.': df_metricas.iloc[:, 1:].std()
    }).T

    return archivo, df_metricas, stats, None

def ejecutar_experimento(nombre_experimento, archivos, n_splits, n_repeats):
    """Orquesta la ejecución paralela y guarda los CSVs resultantes."""
    total_folds = n_splits * n_repeats
    print(f"\n{'='*80}")
    print(f"[INICIO] Experimento: {nombre_experimento} ({total_folds} Folds Totales)")
    print(f"{'='*80}")
    
    resultados = Parallel(n_jobs=-1)(delayed(procesar_dataset)(archivo, n_splits, n_repeats) for archivo in archivos)

    resumen_medias = []
    lista_metricas_crudas = []

    for archivo, df_metricas, stats, error in resultados:
        if error:
            print(error)
            continue
            
        lista_metricas_crudas.append(df_metricas)

        media_esquema = stats.loc['Media'].copy()
        nombre_limpio = archivo.replace('.csv', '').replace('radiomics_', '')
        media_esquema['Esquema'] = nombre_limpio
        resumen_medias.append(media_esquema)

    if resumen_medias:
        # 1. Guardar y mostrar tabla resumen comparativa
        df_final = pd.DataFrame(resumen_medias)
        df_final.set_index('Esquema', inplace=True)
        df_final = df_final.sort_values(by='acc_test', ascending=False)
        
        archivo_resumen = f"resumen_{nombre_experimento}_{total_folds}folds.csv"
        df_final.to_csv(archivo_resumen)
        
        print(f"\n[RESULTADO] Resumen guardado en: {archivo_resumen}")
        print(df_final.round(4).to_string())
        
        # 2. Guardar métricas crudas (necesarias para el test estadístico de mañana)
        df_crudos_total = pd.concat(lista_metricas_crudas)
        archivo_crudos = f"metricas_crudas_{nombre_experimento}_{total_folds}folds.csv"
        df_crudos_total.to_csv(archivo_crudos, index=False)
        print(f"[INFO] Métricas detalladas guardadas en: {archivo_crudos}")

if __name__ == "__main__":
    archivos_datasets = [
        "radiomics_DE_rand_1.csv",
        "radiomics_DE_best_1.csv",
        "radiomics_DE_rand_2.csv",
        "radiomics_DE_best_2.csv",
        "radiomics_Standard_Otsu.csv"
    ]

    print("[INFO] Iniciando Pipeline de Evaluación SVM (Ignorando variables de sesgo)")

    # EXPERIMENTO 1: 10 Folds Clásicos (1 repetición)
    ejecutar_experimento(
        nombre_experimento="Clasico", 
        archivos=archivos_datasets, 
        n_splits=10, 
        n_repeats=1
    )

    # EXPERIMENTO 2: 30 Folds Repetidos (10 splits x 3 repeticiones)
    ejecutar_experimento(
        nombre_experimento="Repetido", 
        archivos=archivos_datasets, 
        n_splits=10, 
        n_repeats=5
    )
    
    ejecutar_experimento(
        nombre_experimento="Repetido",
        archivos=archivos_datasets,
        n_splits=10, 
        n_repeats=3
    )
    
    print("\n[FIN] Todos los experimentos han concluido y los resultados se han guardado exitosamente.")

[INFO] Iniciando Pipeline de Evaluación SVM (Ignorando variables de sesgo)

[INICIO] Experimento: Clasico (10 Folds Totales)

[RESULTADO] Resumen guardado en: resumen_Clasico_10folds.csv
               acc_train  acc_test  recall_test  f1_test  auc_test
Esquema                                                           
DE_best_2         0.9324    0.9322       0.9612   0.9539    0.9595
Standard_Otsu     0.9317    0.9315       0.9600   0.9534    0.9594
DE_rand_1         0.9309    0.9308       0.9590   0.9529    0.9598
DE_best_1         0.9305    0.9305       0.9593   0.9527    0.9587
DE_rand_2         0.9303    0.9302       0.9588   0.9525    0.9600
[INFO] Métricas detalladas guardadas en: metricas_crudas_Clasico_10folds.csv

[INICIO] Experimento: Repetido (50 Folds Totales)

[RESULTADO] Resumen guardado en: resumen_Repetido_50folds.csv
               acc_train  acc_test  recall_test  f1_test  auc_test
Esquema                                                           
DE_best_2         0

## Pruebas estadisticas

In [11]:


def realizar_test_estadistico(archivo_csv, metrica='acc_test'):
    print(f"{'='*60}")
    print(f"[ANÁLISIS ESTADÍSTICO] Evaluando: {archivo_csv}")
    print(f"Métrica objetivo: {metrica}")
    print(f"{'='*60}\n")

    # 1. Cargar datos crudos
    try:
        df = pd.read_csv(archivo_csv)
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo {archivo_csv}.")
        return

    # 2. Reestructurar los datos para el Test de Friedman
    esquemas = df['Esquema'].unique()
    
    # Extraer los arreglos de exactitud para cada esquema asegurando el mismo orden
    datos_por_esquema = []
    for esquema in esquemas:
        valores = df[df['Esquema'] == esquema][metrica].values
        datos_por_esquema.append(valores)

    # Convertir a matriz (Filas: iteraciones, Columnas: algoritmos)
    matriz_datos = np.column_stack(datos_por_esquema)

    # 3. Test de Friedman
    stat, p_value = stats.friedmanchisquare(*datos_por_esquema)
    
    print(f"[TEST DE FRIEDMAN]")
    print(f"Estadístico Chi-cuadrado: {stat:.4f}")
    print(f"P-value: {p_value:.4e}")

    # Evaluar la hipótesis nula (H0: Todos los algoritmos rinden igual)
    alfa = 0.05
    if p_value > alfa:
        print("\n[CONCLUSIÓN] No se rechaza H0.")
        print("No existe evidencia estadística de una diferencia significativa entre los métodos.")
        print("No es necesario realizar la prueba Post-Hoc.")
        return
    else:
        print("\n[CONCLUSIÓN] Se rechaza H0.")
        print("Al menos un método de segmentación tiene un rendimiento significativamente distinto.")
        print("Procediendo con la prueba Post-Hoc de Nemenyi...\n")

    # 4. Prueba Post-Hoc de Nemenyi
    # Nemenyi nos dirá exactamente QUIÉN es mejor que QUIÉN
    print(f"{'-'*60}")
    print(f"[TEST POST-HOC: NEMENYI (P-Values)]")
    print(f"{'-'*60}")
    
    # scikit-posthocs requiere los datos en formato de matriz bidimensional
    p_values_nemenyi = sp.posthoc_nemenyi_friedman(matriz_datos)
    
    # Asignar nombres a filas y columnas para claridad
    p_values_nemenyi.columns = esquemas
    p_values_nemenyi.index = esquemas
    
    print(p_values_nemenyi.round(4).to_string())

    # 5. Visualización: Mapa de calor de P-values
    plt.figure(figsize=(8, 6))
    
    # Crear una máscara para resaltar las diferencias significativas (p < 0.05)
    # Se utiliza un mapa de calor donde los colores oscuros significan diferencia fuerte
    sns.heatmap(p_values_nemenyi, annot=True, cmap="YlGnBu", vmin=0, vmax=0.1, 
                cbar_kws={'label': 'P-value'}, fmt=".4f", linewidths=1)
    
    plt.title(f'Test de Nemenyi: Matriz de P-values\n(Valores < 0.05 indican diferencia significativa)')
    plt.tight_layout()
    
    archivo_grafica = f"heatmap_nemenyi_{archivo_csv.replace('.csv', '')}.png"
    plt.savefig(archivo_grafica, dpi=150)
    plt.close()
    
    print(f"\n[ÉXITO] Mapa de calor guardado como: {archivo_grafica}")

if __name__ == "__main__":
    # Apuntamos directamente al archivo robusto de 30 folds
    archivo_objetivo = "metricas_crudas_Repetido_30folds.csv"
    realizar_test_estadistico(archivo_objetivo, metrica='acc_test')

[ANÁLISIS ESTADÍSTICO] Evaluando: metricas_crudas_Repetido_30folds.csv
Métrica objetivo: acc_test

[TEST DE FRIEDMAN]
Estadístico Chi-cuadrado: 11.9170
P-value: 1.7979e-02

[CONCLUSIÓN] Se rechaza H0.
Al menos un método de segmentación tiene un rendimiento significativamente distinto.
Procediendo con la prueba Post-Hoc de Nemenyi...

------------------------------------------------------------
[TEST POST-HOC: NEMENYI (P-Values)]
------------------------------------------------------------
               DE_rand_1  DE_best_1  DE_rand_2  DE_best_2  Standard_Otsu
DE_rand_1         1.0000     0.9732     0.9579     0.3521         0.6872
DE_best_1         0.9732     1.0000     1.0000     0.1024         0.3072
DE_rand_2         0.9579     1.0000     1.0000     0.0838         0.2657
DE_best_2         0.3521     0.1024     0.0838     1.0000         0.9842
Standard_Otsu     0.6872     0.3072     0.2657     0.9842         1.0000

[ÉXITO] Mapa de calor guardado como: heatmap_nemenyi_metricas_cruda

In [13]:
if __name__ == "__main__":
    # Lista con los dos archivos que generó tu script de SVM
    archivos_a_evaluar = [
        "metricas_crudas_Clasico_10folds.csv",
        "metricas_crudas_Repetido_50folds.csv",
        "metricas_crudas_Repetido_30folds.csv"
    ]
    
    for archivo in archivos_a_evaluar:
        # Ejecutará el análisis completo y generará el mapa de calor para cada uno
        realizar_test_estadistico(archivo, metrica='acc_test')
        print("\n" + "*"*80 + "\n")

[ANÁLISIS ESTADÍSTICO] Evaluando: metricas_crudas_Clasico_10folds.csv
Métrica objetivo: acc_test

[TEST DE FRIEDMAN]
Estadístico Chi-cuadrado: 2.3488
P-value: 6.7189e-01

[CONCLUSIÓN] No se rechaza H0.
No existe evidencia estadística de una diferencia significativa entre los métodos.
No es necesario realizar la prueba Post-Hoc.

********************************************************************************

[ANÁLISIS ESTADÍSTICO] Evaluando: metricas_crudas_Repetido_50folds.csv
Métrica objetivo: acc_test

[TEST DE FRIEDMAN]
Estadístico Chi-cuadrado: 20.8260
P-value: 3.4283e-04

[CONCLUSIÓN] Se rechaza H0.
Al menos un método de segmentación tiene un rendimiento significativamente distinto.
Procediendo con la prueba Post-Hoc de Nemenyi...

------------------------------------------------------------
[TEST POST-HOC: NEMENYI (P-Values)]
------------------------------------------------------------
               DE_rand_1  DE_best_1  DE_rand_2  DE_best_2  Standard_Otsu
DE_rand_1         1.